In [1]:
!pip install selenium


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

all_monuments = []

driver.get("https://egymonuments.gov.eg/en/monuments")

# Waiting for the museum links to load
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-museums a")))
i = 1
while True:
    try:
        # press the next site link
        try:
            site_link = wait.until(EC.element_to_be_clickable(
            (By.XPATH, f'//*[@id="cd-museums"]/app-egyptian-treasure/div/div[3]/a[{i}]')
            ))
            site_link.click()
            i += 1
        except:
            driver.find_element(By.XPATH, '//*[@id="cd-museums"]/app-egyptian-treasure/div/div[3]/div/button').click()
            continue

        # collect the data
        place_name = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.mainPageTitle'))).text
        place_location = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont > div.DescriptionSection > div > div.itemInfo'
        ).text
        place_description = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont > div.DescriptionSection > div > div.txtSection'
        ).text

        try:
            img_elem = driver.find_element(By.CSS_SELECTOR,
                'body > div.innerMainBgCont > div > div:nth-child(1) > div > div.relative.descriptionImageCont > div.imageCont img'
            )
            place_photo = img_elem.get_attribute("src")
        except:
            place_photo = ""

        try:
            start_from = driver.find_element(By.CSS_SELECTOR,
                '#planVisit > div.mapHoursCont > div.openingHoursSec > div:nth-child(2) > div:nth-child(1) > p'
            ).text
            end_at = driver.find_element(By.CSS_SELECTOR,
                '#planVisit > div.mapHoursCont > div.openingHoursSec > div:nth-child(2) > div:nth-child(2) > p'
            ).text
        except:
            start_from, end_at = "Not Available", "Not Available"

        try:
            tickets_price = driver.find_element(By.CSS_SELECTOR, 'div.ticketPriceDetails > div').text
        except:
            tickets_price = "Free"

        all_monuments.append({
            "monument": place_name,
            "location": place_location,
            "Description": place_description,
            "start_from": start_from,
            "end_at": end_at,
            "tickets_price": tickets_price,
            "photo_url": place_photo
        })

        # return to main
        driver.back()
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-museums a")))


    except Exception as e:
        print("Finished — no more items.")
        break

print(f"Total Monuments: {len(all_monuments)}")

Finished — no more items.
Total Monuments: 125


In [4]:
df = pd.DataFrame(all_monuments)
df


,monument,location,Description,start_from,end_at,tickets_price,photo_url
0,Temple of Amenhotep II,Luxor,The Temple of Millions of Years of Amenhotep I...,Not Available,Not Available,Free,https://egymonuments.gov.eg/media/8997/%D8%A7%...
1,Dier El Ballas,Qena,Probably one of the most important sites in Eg...,Not Available,Not Available,Free,https://egymonuments.gov.eg/media/8517/2.jpg?a...
2,khonsu temple,Karnak,"The Temple of Khonsu, nestled in the southwest...",06:00 AM,05:00 PM,Free,https://egymonuments.gov.eg/media/8518/62842d6...
3,The Ramesseum; The Temple of Ramses II,Luxor,"King Ramses II (c.1279-1213 BC), the most famo...",06:00 AM,05:00 PM,FOREIGNERS:\nAdult: EGP 220 / Student: EGP 110...,https://egymonuments.gov.eg/media/8494/whatsap...
4,The Temple of Medinet Habu,Luxor,"Ramses III built his mortuary temple, also kno...",06:00 AM,05:00 PM,FOREIGNERS:\nAdult: EGP 230 / Student: EGP 110...,https://egymonuments.gov.eg/media/8489/whatsap...
...,...,...,...,...,...,...,...
120,Bagawat Cemetery,New Valley Governorate (El Wadi El Gedid Gover...,"Bagawat, near Kharga Oasis, is one of the olde...",09:00 AM,04:00 PM,FOREIGNERS:\nAdult: EGP 120/ StudentS: EGP 60\...,https://egymonuments.gov.eg/media/2360/img_018...
121,Madrasa and Khanaqah of Sultan al-Zahir Barquq,Al-Mu'izz Street,It was established by Sultan al-Zahir Barquq b...,09:00 AM,04:00 PM,FOREIGNERS:\nArea entry:\nAdult: EGP 220 / Stu...,https://egymonuments.gov.eg/media/7428/01.jpg?...
122,Sabil-Kuttab of Abd al-Rahman Katkhuda,Al-Mu'izz Street,This Sabil is located at the intersection of a...,09:00 AM,04:00 PM,FOREIGNERS:\nArea entry:\nAdult: EGP 220 / Stu...,https://egymonuments.gov.eg/media/1298/13.jpg?...
123,Madrasa and Mausoleum of al-Saleh Najm al-Din ...,Al-Mu'izz Street,This is one of the most important architectura...,09:00 AM,04:00 PM,FOREIGNERS:\nArea entry:\nAdult: EGP 220 / Stu...,https://egymonuments.gov.eg/media/1233/%D9%82%...


In [5]:
df.to_csv('monuments_en.csv', index=False)